# NutriBot: Colab training prototype

This notebook fine-tunes a small open-weight language model using the custom starter dataset in this folder. It does not train a model from scratch. The starter dataset is small and synthetic, so this run demonstrates the workflow only; it does not prove nutrition accuracy or clinical safety.

Before running: in Colab choose **Runtime > Change runtime type > T4 GPU** if available. Then run the cells from top to bottom. When asked to upload files, select `train.py`, `nutrition_qa.jsonl`, and `evaluation.jsonl` from the project's `ai-training` folder.

In [ ]:
%pip install -q "transformers>=4.45,<5" "peft>=0.12" "bitsandbytes>=0.43" "accelerate>=1.0"

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

training_dir = Path('/content/nutribot-training')
training_dir.mkdir(parents=True, exist_ok=True)
required_files = {'train.py', 'nutrition_qa.jsonl', 'evaluation.jsonl'}
uploaded = files.upload()
missing = required_files - set(uploaded)
if missing:
    raise ValueError(f"Please upload all required files: {', '.join(sorted(missing))}")
for filename, content in uploaded.items():
    if filename in required_files:
        (training_dir / filename).write_bytes(content)
print('Training files are ready:', ', '.join(sorted(required_files)))

In [ ]:
import json

def validate_jsonl(filename):
    rows = []
    with (training_dir / filename).open(encoding='utf-8') as stream:
        for line_number, line in enumerate(stream, start=1):
            if not line.strip():
                continue
            row = json.loads(line)
            messages = row['messages']
            assert len(messages) == 2 and messages[0]['role'] == 'user' and messages[1]['role'] == 'assistant', f'{filename}:{line_number}'
            target = json.loads(messages[1]['content'])
            assert isinstance(target.get('answer'), str) and isinstance(target.get('needs_fallback'), bool), f'{filename}:{line_number}'
            rows.append(row)
    if not rows:
        raise ValueError(f'{filename} has no examples')
    print(f'{filename}: {len(rows)} valid examples')
    return rows

train_rows = validate_jsonl('nutrition_qa.jsonl')
evaluation_rows = validate_jsonl('evaluation.jsonl')

The evaluation file is held out from training. It currently has only a few examples, so its loss is a basic workflow check—not a meaningful proof of answer quality. Review the generated answers yourself and expand the dataset with reliable, reviewed sources before making claims.

In [ ]:
%cd /content/nutribot-training
!python train.py

In [ ]:
from google.colab import files
import shutil

adapter_dir = Path('/content/nutribot-adapter')
if not adapter_dir.exists() or not any(adapter_dir.iterdir()):
    raise FileNotFoundError('Training output was not found. Check that the training cell completed successfully.')
archive = shutil.make_archive('/content/nutribot-adapter', 'zip', adapter_dir)
print('Download this adapter ZIP and keep it private; do not upload it into the web app source folder.')
files.download(archive)

## Important next steps

This ZIP is a fine-tuned adapter, not a complete standalone model; inference also needs the base checkpoint `Qwen/Qwen2.5-1.5B-Instruct`. The app is not using this model yet. Before connecting it, evaluate a larger, reviewed test set in Filipino, English, and Taglish, including medical-safety and out-of-scope questions. Keep Gemini fallback enabled. Do not place API keys or personal health data in this notebook or training data.